# Clean 690 Generation Experiment

기존 improvement 노트북을 더 덧대지 않고, 공유 README / handoff 문서의 핵심만 반영한 깨끗한 generation 실험용 노트북입니다.

목표:
- 690 retrieval 결과와 690 chunk는 고정합니다.
- Chroma/vector DB는 다시 사용하지 않습니다.
- source_store는 기본으로 끕니다.
- LLM은 짧은 JSON 답변만 생성합니다.
- citation, evidence, failure tag, review 파일은 코드가 붙입니다.
- `wrong_field_choice`, `unanswerable_mishandled`를 줄이는 데 집중합니다.


In [4]:
# 1. Experiment config
from pathlib import Path

REPO_URL = 'https://github.com/beomsookim1020/chatbot.git'
REPO_BRANCH = 'colab-generation'
PROJECT_DIR = Path('/content/chatbot')

DRIVE_INPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_inputs')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_outputs')
DRIVE_EXPERIMENT_ROOT = DRIVE_OUTPUT_ROOT / 'generation_clean_handoff_experiments'

PREDICTION_REL = Path(
    'outputs/predictions/91_dense_qdecomp_rrf_per75_docscore_mean3_300_kure_chroma_690_canonical.jsonl'
)
CHUNK_SIDECAR_REL = Path('indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl')
EXPERIMENT_EVAL_FILENAME = 'representative_wrong_30_eval_batch_format.csv'

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
RUN_LIMIT = 0
EXPERIMENT_IDS = []
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0
TOP_P = 1.0

TOP_CONTEXTS = 6
TOTAL_RAW_CONTEXT_CHARS = 9000
PER_CONTEXT_MAX_CHARS = 2200
EVIDENCE_PER_INTENT = 8

EXPERIMENT_NAME = 'clean_690_field_aware_json_answer'
USE_SOURCE_STORE = False

print('PROJECT_DIR:', PROJECT_DIR)
print('prediction:', PREDICTION_REL)
print('chunks:', CHUNK_SIDECAR_REL)
print('eval:', EXPERIMENT_EVAL_FILENAME)
print('model:', MODEL_NAME)


PROJECT_DIR: /content/chatbot
prediction: outputs/predictions/91_dense_qdecomp_rrf_per75_docscore_mean3_300_kure_chroma_690_canonical.jsonl
chunks: indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl
eval: representative_wrong_30_eval_batch_format.csv
model: Qwen/Qwen2.5-3B-Instruct


In [5]:
# 2. Runtime setup: GPU, Drive, repo, minimal packages
import shutil
import subprocess
import sys

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Colab GPU runtime이 아닙니다. Runtime > Change runtime type > GPU로 바꿔주세요.')
subprocess.run(['nvidia-smi'], check=True)

from google.colab import drive
drive.mount('/content/drive')

def run_cmd(cmd, cwd=None):
    print('$', ' '.join(map(str, cmd)))
    subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, check=True)

if (PROJECT_DIR / '.git').exists():
    run_cmd(['git', 'fetch', 'origin', REPO_BRANCH], cwd=PROJECT_DIR)
    run_cmd(['git', 'checkout', REPO_BRANCH], cwd=PROJECT_DIR)
    run_cmd(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=PROJECT_DIR)
else:
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    run_cmd(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, PROJECT_DIR])

# pandas/numpy ABI 문제를 피하려고 generation에 필요한 최소 패키지만 설치합니다.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.45.0', 'accelerate', 'sentencepiece', 'safetensors'
], check=True)

ROOT = str(PROJECT_DIR)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print('Runtime ready.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git fetch origin colab-generation
$ git checkout colab-generation
$ git pull --ff-only origin colab-generation
Runtime ready.


In [6]:
# 3. Copy inputs from Drive to Colab local
import shutil

DRIVE_EVAL_FILE = DRIVE_INPUT_ROOT / 'data/eval' / EXPERIMENT_EVAL_FILENAME
DRIVE_PREDICTIONS = DRIVE_INPUT_ROOT / PREDICTION_REL
chunk_candidates = [
    DRIVE_INPUT_ROOT / CHUNK_SIDECAR_REL,
    (DRIVE_INPUT_ROOT / CHUNK_SIDECAR_REL).with_suffix('.json'),
]
DRIVE_CHUNKS = next((path for path in chunk_candidates if path.exists()), None)

missing = [str(path) for path in [DRIVE_EVAL_FILE, DRIVE_PREDICTIONS] if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required inputs:\n' + '\n'.join(missing))
if DRIVE_CHUNKS is None:
    print('WARN: chunk sidecar를 찾지 못했습니다. clean 노트북은 retrieval JSONL의 retrieved_contexts만 사용하므로 계속 진행합니다.')

LOCAL_EVAL_DIR = PROJECT_DIR / 'data/experiment_eval_clean'
LOCAL_EVAL_FILE = LOCAL_EVAL_DIR / EXPERIMENT_EVAL_FILENAME
LOCAL_PREDICTIONS = PROJECT_DIR / PREDICTION_REL
LOCAL_CHUNKS = PROJECT_DIR / CHUNK_SIDECAR_REL if DRIVE_CHUNKS else None

if LOCAL_EVAL_DIR.exists():
    shutil.rmtree(LOCAL_EVAL_DIR)
LOCAL_EVAL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_PREDICTIONS.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_CHUNKS:
    LOCAL_CHUNKS.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(DRIVE_EVAL_FILE, LOCAL_EVAL_FILE)
shutil.copy2(DRIVE_PREDICTIONS, LOCAL_PREDICTIONS)
if DRIVE_CHUNKS and LOCAL_CHUNKS:
    shutil.copy2(DRIVE_CHUNKS, LOCAL_CHUNKS)

print('eval:', LOCAL_EVAL_FILE)
print('predictions:', LOCAL_PREDICTIONS)
print('chunks:', LOCAL_CHUNKS)


eval: /content/chatbot/data/experiment_eval_clean/representative_wrong_30_eval_batch_format.csv
predictions: /content/chatbot/outputs/predictions/91_dense_qdecomp_rrf_per75_docscore_mean3_300_kure_chroma_690_canonical.jsonl
chunks: /content/chatbot/indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl


In [7]:
# 4. Load eval rows and retrieval predictions without pandas
import ast
import csv
import json
import re
import time
from collections import Counter
from datetime import datetime, timezone
from html import escape
from pathlib import Path
from typing import Any

from src.generator import HuggingFaceGenerator

def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open('r', encoding='utf-8') as file:
        for line_no, line in enumerate(file, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'Invalid JSONL at {path}:{line_no}: {exc}') from exc
    return rows

def read_csv_records(path: Path) -> list[dict[str, str]]:
    with path.open('r', encoding='utf-8-sig', newline='') as file:
        return list(csv.DictReader(file))

def parse_structured(value, default):
    if value in (None, ''):
        return default
    if isinstance(value, (list, dict)):
        return value
    text = str(value).strip()
    for parser in (json.loads, ast.literal_eval):
        try:
            return parser(text)
        except Exception:
            pass
    return default

def parse_doc_list(value):
    parsed = parse_structured(value, [])
    if isinstance(parsed, list):
        return [str(item) for item in parsed]
    if isinstance(parsed, str) and parsed.strip():
        return [part.strip() for part in re.split(r'[,;]', parsed) if part.strip()]
    return []

def first_non_empty(row, *keys):
    for key in keys:
        value = row.get(key)
        if value not in (None, ''):
            return value
    return None

def normalize_retrieved_contexts(value):
    contexts = parse_structured(value, [])
    if not isinstance(contexts, list):
        return []
    return [context for context in contexts if isinstance(context, dict)]

eval_rows = read_csv_records(LOCAL_EVAL_FILE)
pred_rows = read_jsonl(LOCAL_PREDICTIONS)
pred_by_id = {str(row.get('id')): row for row in pred_rows}

rows = []
for eval_row in eval_rows:
    qid = str(eval_row.get('id'))
    pred = pred_by_id.get(qid)
    if not pred:
        continue
    rows.append({
        'id': qid,
        'type': eval_row.get('type'),
        'difficulty': eval_row.get('difficulty'),
        'question': first_non_empty(eval_row, 'question', 'question_eval') or pred.get('question'),
        'ground_truth_answer': eval_row.get('ground_truth_answer'),
        'ground_truth_docs': parse_doc_list(eval_row.get('ground_truth_docs')),
        'metadata_filter': parse_structured(eval_row.get('metadata_filter'), {}),
        'retrieved_contexts': normalize_retrieved_contexts(pred.get('retrieved_contexts')),
    })

if EXPERIMENT_IDS:
    keep = set(EXPERIMENT_IDS)
    rows = [row for row in rows if row['id'] in keep]
elif RUN_LIMIT and RUN_LIMIT > 0:
    rows = rows[:RUN_LIMIT]

print('eval rows:', len(eval_rows))
print('prediction rows:', len(pred_rows))
print('selected rows:', len(rows))
print('first ids:', [row['id'] for row in rows[:5]])


eval rows: 30
prediction rows: 500
selected rows: 30
first ids: ['Q006', 'Q017', 'Q020', 'Q031', 'Q123']


In [8]:
# 5. Clean field-aware context builder
MONEY_PATTERN = re.compile(r'(?:금\s*)?(?:[0-9]{1,3}(?:,[0-9]{3})+|[0-9]+(?:\.[0-9]+)?)\s*(?:원|천원|만원|억원|백만원|KRW|부가세|VAT)|KRW\s*:?\s*[0-9,]+')
DATE_PATTERN = re.compile(r'(?:20\d{2}|\d{2})[.\-/년]\s*\d{1,2}[.\-/월]\s*\d{1,2}(?:일)?|\d{1,2}\s*월\s*\d{1,2}\s*일|\d{1,2}:\d{2}')

BUDGET_WORDS = ('예산', '사업비', '사업예산', '금액', '기초금액', '추정가격', '예정가격', '배정액', '총사업비', '용역비', '가격', '원')
DATE_WORDS = ('기간', '마감', '제출', '접수', '공고일', '입찰', '계약기간', '개찰', '기한')
SUBMISSION_WORDS = ('제출서류', '제출 서류', '구비서류', '제안서', '입찰참가', '첨부', '서식', '파일')
QUALIFICATION_WORDS = ('자격', '참가자격', '입찰자격', '요건', '제한', '실적', '면허', '등록')
PURPOSE_WORDS = ('목적', '배경', '개요', '내용', '범위', '과업', '추진', '대상', '서비스', '지원')
NEGATIVE_WORDS = ('없는', '없나요', '없는지', '요구하지', '제외', '해당하지')

BUDGET_ALLOWED_LABELS = {
    'total_allocation': ('전체 배정액', '배정액'),
    'project_budget': ('사업예산', '예산액', '사업비', '총사업비', '용역비'),
    'estimated_price': ('추정가격', '추정 금액'),
    'base_amount': ('기초금액', '기초 금액'),
    'expected_price': ('예정가격', '예정 가격'),
}
BUDGET_BLOCKED_WORDS = ('보증금', '입찰보증', '계약보증', '하자보증', '지급조건', '선금', '중도금', '잔금', '위약', '벌금', '배점', '점수', '평가점수')

def coerce_text(value):
    return '' if value is None else str(value)

def unique(values, limit=30):
    out = []
    for value in values:
        text = re.sub(r'\s+', ' ', coerce_text(value)).strip()
        if text and text not in out:
            out.append(text)
        if len(out) >= limit:
            break
    return out

def parse_krw(text):
    text = coerce_text(text).replace(',', '').replace(' ', '')
    match = re.search(r'(\d+(?:\.\d+)?)', text)
    if not match:
        return None
    value = float(match.group(1))
    if '억원' in text:
        value *= 100_000_000
    elif '백만원' in text:
        value *= 1_000_000
    elif '천원' in text:
        value *= 1_000
    elif '만원' in text:
        value *= 10_000
    return int(value)

def classify_question_clean(question):
    q = coerce_text(question)
    intents = []
    if any(word in q for word in BUDGET_WORDS):
        question_type = 'budget'
        if any(word in q for word in ('차액', '차이', '얼마 차', '비교')):
            intents.append('budget_difference')
        elif any(word in q for word in ('합계', '합산', '총액', '총 예산')):
            intents.append('budget_sum')
        elif any(word in q for word in ('비율', '퍼센트', '%', '나누', '단가')):
            intents.append('budget_ratio')
        else:
            intents.append('budget_lookup')
    elif any(word in q for word in SUBMISSION_WORDS):
        question_type = 'submission_documents'
        intents.append('submission_documents')
    elif any(word in q for word in DATE_WORDS):
        question_type = 'date_or_period'
        intents.append('date_or_period')
    elif any(word in q for word in QUALIFICATION_WORDS):
        question_type = 'qualification'
        intents.append('qualification')
    else:
        question_type = 'general'
        intents.append('general_answer')
    if any(word in q for word in PURPOSE_WORDS) and 'purpose_summary' not in intents:
        intents.append('purpose_summary')
    if any(word in q for word in NEGATIVE_WORDS):
        intents.append('negative_check')
    return {'question_type': question_type, 'intent_slots': intents}

def normalize_contexts(contexts):
    records = []
    for rank, context in enumerate(contexts[:TOP_CONTEXTS], 1):
        metadata = context.get('metadata') or {}
        text = coerce_text(context.get('text') or context.get('content') or context.get('page_content'))
        records.append({
            'rank': context.get('rank') or rank,
            'filename': context.get('filename') or metadata.get('source_file') or '',
            'doc_id': context.get('doc_id') or metadata.get('doc_id') or '',
            'chunk_id': context.get('chunk_id') or metadata.get('chunk_id') or '',
            'score': context.get('score'),
            'metadata': metadata,
            'text': text,
        })
    return records

def wanted_budget_types(question):
    q = coerce_text(question)
    wanted = []
    if '전체 배정액' in q or '배정액' in q:
        wanted.append('total_allocation')
    if '추정가격' in q:
        wanted.append('estimated_price')
    if '기초금액' in q:
        wanted.append('base_amount')
    if '예정가격' in q:
        wanted.append('expected_price')
    if any(word in q for word in ('사업예산', '사업비', '예산')):
        wanted.append('project_budget')
    return wanted or ['project_budget', 'total_allocation', 'estimated_price', 'base_amount', 'expected_price']

def budget_type_for_line(line):
    for budget_type, labels in BUDGET_ALLOWED_LABELS.items():
        if any(label in line for label in labels):
            return budget_type
    return 'unknown_budget_amount'

def extract_line_candidates(records, question, analysis):
    lines = []
    for record in records:
        raw_lines = re.split(r'\n+|(?<=[.!?。])\s+', record['text'])
        for line_no, raw in enumerate(raw_lines, 1):
            line = re.sub(r'\s+', ' ', raw).strip()
            if line:
                lines.append((record, line_no, line[:700]))

    wanted_types = wanted_budget_types(question)
    candidates = {'budget_allowed': [], 'budget_blocked': [], 'budget_neutral': [], 'dates': [], 'submission': [], 'qualification': [], 'purpose': []}
    evidence = []

    for record, line_no, line in lines:
        base = {'rank': record['rank'], 'filename': record['filename'], 'chunk_id': record['chunk_id'], 'line_no': line_no, 'text': line}
        if MONEY_PATTERN.search(line):
            item = dict(base)
            item['amounts'] = MONEY_PATTERN.findall(line)
            item['budget_type'] = budget_type_for_line(line)
            item['blocked_reason'] = next((word for word in BUDGET_BLOCKED_WORDS if word in line), '')
            item['type_match'] = item['budget_type'] in wanted_types
            idx = len(candidates['budget_allowed']) + len(candidates['budget_blocked']) + len(candidates['budget_neutral']) + 1
            item['candidate_id'] = f'B{idx:02d}'
            if item['blocked_reason']:
                candidates['budget_blocked'].append(item)
            elif item['budget_type'] != 'unknown_budget_amount':
                candidates['budget_allowed'].append(item)
            else:
                candidates['budget_neutral'].append(item)
        if DATE_PATTERN.search(line) or any(word in line for word in DATE_WORDS):
            item = dict(base, candidate_id=f"D{len(candidates['dates']) + 1:02d}", dates=DATE_PATTERN.findall(line))
            candidates['dates'].append(item)
        if any(word in line for word in SUBMISSION_WORDS):
            item = dict(base, candidate_id=f"S{len(candidates['submission']) + 1:02d}")
            candidates['submission'].append(item)
        if any(word in line for word in QUALIFICATION_WORDS):
            item = dict(base, candidate_id=f"Q{len(candidates['qualification']) + 1:02d}")
            candidates['qualification'].append(item)
        if any(word in line for word in PURPOSE_WORDS):
            item = dict(base, candidate_id=f"P{len(candidates['purpose']) + 1:02d}")
            candidates['purpose'].append(item)

    for key in candidates:
        candidates[key] = candidates[key][:20]

    if analysis['question_type'] == 'budget':
        evidence.extend(sorted(candidates['budget_allowed'], key=lambda x: (not x.get('type_match'), x['rank']))[:EVIDENCE_PER_INTENT])
        if not evidence:
            evidence.extend(candidates['budget_neutral'][:EVIDENCE_PER_INTENT])
    elif analysis['question_type'] == 'date_or_period':
        evidence.extend(candidates['dates'][:EVIDENCE_PER_INTENT])
    elif analysis['question_type'] == 'submission_documents':
        evidence.extend(candidates['submission'][:EVIDENCE_PER_INTENT])
    elif analysis['question_type'] == 'qualification':
        evidence.extend(candidates['qualification'][:EVIDENCE_PER_INTENT])
    else:
        evidence.extend(candidates['purpose'][:EVIDENCE_PER_INTENT])

    if 'purpose_summary' in analysis['intent_slots']:
        evidence.extend(candidates['purpose'][:4])
    if 'negative_check' in analysis['intent_slots']:
        evidence.extend(candidates['submission'][:3] + candidates['qualification'][:3] + candidates['purpose'][:3])

    seen = set()
    deduped = []
    for item in evidence:
        key = (item.get('filename'), item.get('chunk_id'), item.get('text'))
        if key not in seen:
            seen.add(key)
            deduped.append(item)
    return candidates, deduped[:12]

def deterministic_budget_value(question, candidates, analysis):
    allowed = sorted(candidates.get('budget_allowed') or [], key=lambda x: (not x.get('type_match'), x['rank']))
    usable = allowed or candidates.get('budget_neutral') or []
    values = []
    for item in usable:
        for raw in item.get('amounts') or []:
            amount = parse_krw(raw)
            if amount:
                values.append({'raw': raw, 'amount': amount, 'candidate_id': item['candidate_id'], 'budget_type': item.get('budget_type'), 'filename': item.get('filename'), 'text': item.get('text')})
    if not values:
        return None
    intents = set(analysis['intent_slots'])
    if 'budget_sum' in intents and len(values) >= 2:
        return {'kind': 'budget_sum', 'raw': f"{sum(v['amount'] for v in values[:4]):,}원", 'operands': values[:4]}
    if 'budget_difference' in intents and len(values) >= 2:
        diff = abs(values[0]['amount'] - values[1]['amount'])
        return {'kind': 'budget_difference', 'raw': f'{diff:,}원', 'operands': values[:2]}
    return {'kind': 'budget_lookup', 'raw': values[0]['raw'], 'operands': values[:1]}

def short_candidate(item):
    keep = ['candidate_id', 'filename', 'chunk_id', 'budget_type', 'amounts', 'dates', 'type_match', 'blocked_reason', 'text']
    return {key: item.get(key) for key in keep if item.get(key) not in (None, '', [])}

def render_context_package(row):
    question = row['question']
    analysis = classify_question_clean(question)
    records = normalize_contexts(row['retrieved_contexts'])
    candidates, evidence = extract_line_candidates(records, question, analysis)
    computed = deterministic_budget_value(question, candidates, analysis) if analysis['question_type'] == 'budget' else None

    raw_parts = []
    total = 0
    for record in records:
        text = record['text'][:PER_CONTEXT_MAX_CHARS]
        if total + len(text) > TOTAL_RAW_CONTEXT_CHARS:
            text = text[:max(0, TOTAL_RAW_CONTEXT_CHARS - total)]
        if not text:
            break
        total += len(text)
        raw_parts.append(f"--- rank {record['rank']} | {record['filename']} | {record['chunk_id']} ---\n{text}")

    return {'question': question, 'analysis': analysis, 'context_records': records, 'field_candidates': candidates, 'selected_evidence': evidence, 'computed_values': computed, 'raw_context': '\n\n'.join(raw_parts)}

def build_prompt(package):
    analysis = package['analysis']
    compact_candidates = {key: [short_candidate(item) for item in values[:8]] for key, values in package['field_candidates'].items() if values}
    evidence = [short_candidate(item) for item in package['selected_evidence']]
    computed = package.get('computed_values')
    return f"""너는 RFP 문서 기반 QA assistant다.

너의 역할은 답변 JSON만 짧게 생성하는 것이다. citation, failure tag, 검수 파일은 코드가 따로 붙인다.

[핵심 규칙]
1. 제공된 Context Package 안의 정보만 사용한다.
2. `answer_status`는 `answered`, `ambiguous`, `not_found_in_context` 중 하나만 쓴다.
3. 후보값이나 근거 문장이 있으면 바로 `not_found_in_context`로 답하지 말고 먼저 후보를 검토한다.
4. 예산 질문에서는 `budget_allowed` 후보를 우선 사용하고, `budget_blocked` 후보는 최종 답으로 쓰지 않는다.
5. 여러 후보가 충돌하면 `ambiguous`로 두고 후보 차이를 설명한다.
6. 차액/합계 계산값이 `[computed_values]`에 있으면 그 값을 우선 사용한다.
7. 답변에 사용한 후보가 있으면 `used_candidate_ids`에 candidate_id를 넣는다.
8. JSON 외의 텍스트는 출력하지 않는다.

[출력 JSON schema]
{{
  "answer_status": "answered|ambiguous|not_found_in_context",
  "answer": "짧은 한국어 답변",
  "used_candidate_ids": ["B01"],
  "reason": "왜 이 후보를 썼는지 한 문장"
}}

[question]
{package['question']}

[intent_plan]
{json.dumps(analysis, ensure_ascii=False, indent=2)}

[computed_values]
{json.dumps(computed or {}, ensure_ascii=False, indent=2)}

[field_candidates]
{json.dumps(compact_candidates, ensure_ascii=False, indent=2)}

[selected_evidence]
{json.dumps(evidence, ensure_ascii=False, indent=2)}

[raw_retrieval_context]
{package['raw_context']}
""".strip()


In [9]:
# 6. LLM output parsing, diagnostics, and save helpers
def extract_json_object(text):
    text = coerce_text(text).strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.DOTALL).strip()
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            return data, True, None
    except Exception as exc:
        parse_error = type(exc).__name__
    start = text.find('{')
    end = text.rfind('}')
    if start >= 0 and end > start:
        try:
            data = json.loads(text[start:end + 1])
            if isinstance(data, dict):
                return data, False, 'recovered_outer_json'
        except Exception as exc:
            parse_error = type(exc).__name__
    return {'answer_status': 'ambiguous', 'answer': text[:1000], 'used_candidate_ids': [], 'reason': 'invalid JSON recovered as raw text'}, False, parse_error

def normalize_answer(parsed):
    status = parsed.get('answer_status') or 'ambiguous'
    if status not in {'answered', 'ambiguous', 'not_found_in_context'}:
        status = 'ambiguous'
    used = parsed.get('used_candidate_ids') or []
    if isinstance(used, str):
        used = [used]
    return {'answer_status': status, 'answer': coerce_text(parsed.get('answer')).strip(), 'used_candidate_ids': [coerce_text(item).strip() for item in used if coerce_text(item).strip()], 'reason': coerce_text(parsed.get('reason')).strip()}

def candidate_lookup(package):
    lookup = {}
    for values in package['field_candidates'].values():
        for item in values:
            lookup[item.get('candidate_id')] = item
    return lookup

def attach_citations(answer_obj, package):
    lookup = candidate_lookup(package)
    citations = []
    for cid in answer_obj.get('used_candidate_ids') or []:
        item = lookup.get(cid)
        if item:
            citations.append({'candidate_id': cid, 'filename': item.get('filename'), 'chunk_id': item.get('chunk_id'), 'evidence_text': item.get('text')})
    if not citations:
        for item in package.get('selected_evidence') or []:
            citations.append({'candidate_id': item.get('candidate_id'), 'filename': item.get('filename'), 'chunk_id': item.get('chunk_id'), 'evidence_text': item.get('text')})
            if len(citations) >= 2:
                break
    return citations

def diagnose(record):
    answer = record['answer']['answer']
    status = record['answer']['answer_status']
    package = record['context_package']
    candidates = package['field_candidates']
    context_text = package['raw_context'] + '\n' + json.dumps(candidates, ensure_ascii=False)
    tags = []
    if not record['_valid_json']:
        tags.append('llm_invalid_json_recovered')
    missing = [token for token in MONEY_PATTERN.findall(answer) + DATE_PATTERN.findall(answer) if token and token not in context_text]
    if missing:
        tags.append('answer_number_not_in_context')
    if status == 'not_found_in_context' and any(candidates.get(key) for key in candidates):
        tags.append('unanswerable_mishandled_risk')
    if package['analysis']['question_type'] == 'budget':
        blocked_amounts = []
        for item in candidates.get('budget_blocked') or []:
            blocked_amounts.extend(item.get('amounts') or [])
        if any(amount and amount in answer for amount in blocked_amounts):
            tags.append('wrong_field_choice_risk')
        if not candidates.get('budget_allowed') and candidates.get('budget_neutral'):
            tags.append('source_numeric_found_but_type_uncertain')
        if not candidates.get('budget_allowed') and not candidates.get('budget_neutral'):
            tags.append('source_numeric_missing')
    if len(package['analysis'].get('intent_slots') or []) > 1 and status == 'answered' and len(answer) < 30:
        tags.append('multi_intent_incomplete_risk')
    if not record.get('citations'):
        tags.append('citation_missing')
    return sorted(set(tags)), missing

def build_review_row(record):
    package = record['context_package']
    return {
        'id': record['id'],
        'question_type': package['analysis']['question_type'],
        'intent_slots': json.dumps(package['analysis']['intent_slots'], ensure_ascii=False),
        'question': record['question'],
        'answer_status': record['answer']['answer_status'],
        'generated_answer': record['answer']['answer'],
        'ground_truth_answer': record.get('ground_truth_answer'),
        'ground_truth_docs': json.dumps(record.get('ground_truth_docs'), ensure_ascii=False),
        'used_candidate_ids': json.dumps(record['answer'].get('used_candidate_ids'), ensure_ascii=False),
        'citations': json.dumps(record.get('citations'), ensure_ascii=False),
        'failure_tags': json.dumps(record.get('_failure_tags'), ensure_ascii=False),
        'budget_allowed': json.dumps([short_candidate(x) for x in package['field_candidates'].get('budget_allowed', [])[:5]], ensure_ascii=False),
        'budget_blocked': json.dumps([short_candidate(x) for x in package['field_candidates'].get('budget_blocked', [])[:5]], ensure_ascii=False),
        'human_correctness': '',
        'evidence_grounded': '',
        'failure_type': '',
        'review_memo': '',
    }

def write_jsonl(path, records):
    with path.open('w', encoding='utf-8') as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + '\n')

def write_csv(path, rows):
    if not rows:
        return
    with path.open('w', encoding='utf-8-sig', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

def write_review_html(path, review_rows):
    parts = ['<html><head><meta charset="utf-8"><style>body{font-family:Arial,sans-serif} table{border-collapse:collapse;width:100%} td,th{border:1px solid #ddd;padding:6px;vertical-align:top} th{background:#f4f4f4} .answer{white-space:pre-wrap}</style></head><body>']
    parts.append('<h1>Clean 690 Generation Review</h1>')
    parts.append('<table><thead><tr><th>ID</th><th>Question</th><th>Answer</th><th>GT</th><th>Tags</th><th>Citations</th></tr></thead><tbody>')
    for row in review_rows:
        parts.append('<tr>')
        parts.append(f"<td>{escape(row['id'])}<br>{escape(row['question_type'])}</td>")
        parts.append(f"<td>{escape(row['question'])}</td>")
        parts.append(f"<td class='answer'><b>{escape(row['answer_status'])}</b><br>{escape(row['generated_answer'])}</td>")
        parts.append(f"<td>{escape(coerce_text(row['ground_truth_answer']))}</td>")
        parts.append(f"<td>{escape(row['failure_tags'])}</td>")
        parts.append(f"<td>{escape(row['citations'])}</td>")
        parts.append('</tr>')
    parts.append('</tbody></table></body></html>')
    path.write_text('\n'.join(parts), encoding='utf-8')


In [10]:
# 7. Run generation
RUN_TAG = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
LOCAL_OUTPUT_DIR = PROJECT_DIR / 'outputs/generation_clean_handoff' / RUN_TAG
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

generator = HuggingFaceGenerator(
    model_name=MODEL_NAME,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
)

records = []
for index, row in enumerate(rows, 1):
    started = time.perf_counter()
    package = render_context_package(row)
    prompt = build_prompt(package)
    raw_text = generator.generate_prompt(prompt)
    parsed, valid_json, parse_error = extract_json_object(raw_text)
    answer_obj = normalize_answer(parsed)
    citations = attach_citations(answer_obj, package)
    record = {
        'id': row['id'],
        'question': row['question'],
        'ground_truth_answer': row.get('ground_truth_answer'),
        'ground_truth_docs': row.get('ground_truth_docs'),
        'model': MODEL_NAME,
        'experiment_name': EXPERIMENT_NAME,
        'answer': answer_obj,
        'raw_llm_output': raw_text,
        'citations': citations,
        'context_package': package,
        '_valid_json': valid_json,
        '_parse_error_type': parse_error,
        'latency_ms': int((time.perf_counter() - started) * 1000),
    }
    tags, unsupported = diagnose(record)
    record['_failure_tags'] = tags
    record['_unsupported_numbers_or_dates'] = unsupported
    records.append(record)
    print(f"{index}/{len(rows)} {row['id']} status={answer_obj['answer_status']} tags={tags}")

print('Local output dir:', LOCAL_OUTPUT_DIR)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


1/30 Q006 status=answered tags=[]
2/30 Q017 status=ambiguous tags=['citation_missing', 'source_numeric_missing']
3/30 Q020 status=not_found_in_context tags=['citation_missing', 'source_numeric_missing']
4/30 Q031 status=ambiguous tags=['source_numeric_missing']
5/30 Q123 status=answered tags=[]
6/30 Q137 status=not_found_in_context tags=['unanswerable_mishandled_risk']
7/30 Q001 status=ambiguous tags=['source_numeric_missing']
8/30 Q013 status=ambiguous tags=['source_numeric_missing']
9/30 Q076 status=answered tags=['source_numeric_missing']
10/30 Q104 status=answered tags=[]
11/30 Q088 status=answered tags=['answer_number_not_in_context']
12/30 Q007 status=answered tags=['wrong_field_choice_risk']
13/30 Q008 status=answered tags=[]
14/30 Q016 status=answered tags=['answer_number_not_in_context']
15/30 Q026 status=answered tags=['answer_number_not_in_context', 'multi_intent_incomplete_risk']
16/30 Q028 status=answered tags=[]
17/30 Q003 status=answered tags=[]
18/30 Q061 status=answere

In [11]:
# 8. Save outputs and copy to Drive
review_rows = [build_review_row(record) for record in records]
failure_counter = Counter(tag for record in records for tag in record.get('_failure_tags', []))
status_counter = Counter(record['answer']['answer_status'] for record in records)

metrics_summary = {
    'experiment_name': EXPERIMENT_NAME,
    'model': MODEL_NAME,
    'prediction': str(PREDICTION_REL),
    'chunks': str(CHUNK_SIDECAR_REL),
    'eval_file': EXPERIMENT_EVAL_FILENAME,
    'rows': len(records),
    'valid_json_rate': sum(1 for r in records if r['_valid_json']) / max(len(records), 1),
    'answer_status_counts': dict(status_counter),
    'failure_tag_counts': dict(failure_counter),
    'avg_latency_ms': int(sum(r.get('latency_ms', 0) for r in records) / max(len(records), 1)),
    'use_source_store': USE_SOURCE_STORE,
}

write_jsonl(LOCAL_OUTPUT_DIR / 'generated_answers.jsonl', records)
write_csv(LOCAL_OUTPUT_DIR / 'review_samples.csv', review_rows)
write_csv(LOCAL_OUTPUT_DIR / 'llm_answer_review.csv', review_rows)
write_review_html(LOCAL_OUTPUT_DIR / 'llm_answer_review.html', review_rows)
(LOCAL_OUTPUT_DIR / 'metrics_summary.json').write_text(json.dumps(metrics_summary, ensure_ascii=False, indent=2), encoding='utf-8')
(LOCAL_OUTPUT_DIR / 'failure_tags_summary.json').write_text(json.dumps(dict(failure_counter), ensure_ascii=False, indent=2), encoding='utf-8')

summary_md = f'''# Clean 690 Generation Experiment Summary

- experiment: `{EXPERIMENT_NAME}`
- model: `{MODEL_NAME}`
- rows: `{len(records)}`
- prediction: `{PREDICTION_REL}`
- chunks: `{CHUNK_SIDECAR_REL}`
- source_store: `{USE_SOURCE_STORE}`

## What Changed

- LLM은 짧은 JSON 답변만 생성합니다.
- 예산 후보를 `allowed / blocked / neutral`로 분리합니다.
- 후보가 있는데도 확인 불가로 답하는 경우를 `unanswerable_mishandled_risk`로 표시합니다.
- blocked 예산 후보를 답으로 쓰면 `wrong_field_choice_risk`로 표시합니다.
- citation/evidence/failure tag는 코드가 붙입니다.

## Metrics

```json
{json.dumps(metrics_summary, ensure_ascii=False, indent=2)}
```

## Review Files

- `review_samples.csv`
- `llm_answer_review.csv`
- `llm_answer_review.html`
- `generated_answers.jsonl`
'''
(LOCAL_OUTPUT_DIR / 'experiment_summary.md').write_text(summary_md, encoding='utf-8')

DRIVE_OUTPUT_DIR = DRIVE_EXPERIMENT_ROOT / RUN_TAG
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for path in sorted(LOCAL_OUTPUT_DIR.glob('*')):
    if path.suffix.lower() in {'.jsonl', '.csv', '.html', '.json', '.md'}:
        shutil.copy2(path, DRIVE_OUTPUT_DIR / path.name)

print('Local output dir:', LOCAL_OUTPUT_DIR)
print('Drive output dir:', DRIVE_OUTPUT_DIR)
print(json.dumps(metrics_summary, ensure_ascii=False, indent=2))


Local output dir: /content/chatbot/outputs/generation_clean_handoff/20260527_044717
Drive output dir: /content/drive/MyDrive/chatbot_colab_outputs/generation_clean_handoff_experiments/20260527_044717
{
  "experiment_name": "clean_690_field_aware_json_answer",
  "model": "Qwen/Qwen2.5-3B-Instruct",
  "prediction": "outputs/predictions/91_dense_qdecomp_rrf_per75_docscore_mean3_300_kure_chroma_690_canonical.jsonl",
  "chunks": "indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl",
  "eval_file": "representative_wrong_30_eval_batch_format.csv",
  "rows": 30,
  "valid_json_rate": 1.0,
  "answer_status_counts": {
    "answered": 19,
    "ambiguous": 6,
    "not_found_in_context": 5
  },
  "failure_tag_counts": {
    "citation_missing": 4,
    "source_numeric_missing": 8,
    "unanswerable_mishandled_risk": 3,
    "answer_number_not_in_context": 4,
    "wrong_field_choice_risk": 1,
    "multi_intent_incomplete_risk": 1,
    "source_numeric_found_but_type_uncertain": 2
  },
  "

## 결과 보는 순서

1. `experiment_summary.md`에서 설정과 failure tag 분포를 봅니다.
2. `llm_answer_review.html`에서 질문별 답변/GT/failure tag/citation을 빠르게 봅니다.
3. `review_samples.csv`에서 사람이 `human_correctness`, `evidence_grounded`, `failure_type`, `review_memo`를 채웁니다.
4. `wrong_field_choice_risk`와 `unanswerable_mishandled_risk`가 줄었는지 먼저 확인합니다.

이번 노트북은 새 기준선입니다. 여기서 결과가 안정되면 그 다음에만 source_store, 모델 변경, Judge LLM을 하나씩 붙입니다.
